# Create PRIN Awards (Italy, MUR / MIUR — Progetti di Rilevante Interesse Nazionale)

Creates PRIN grant awards from the research ministry's own public PRIN portal
(CINECA-hosted) at https://prin.mur.gov.it/ — the "Cerca un progetto" search,
which needs **no login** and covers the PRIN calls **1999–2020**. Each
project's public detail fragment (`/Progetti/GetDettaglio?codice=<code>`)
gives: protocol code, lead institution, coordinator/PI, title, awarded amount
(EUR; pre-2002 calls already converted from lire by the portal), duration in
months, ERC panel / CUN area, and the research units (Sede N: institution,
unit PI, unit amount). The researcher login area is not used.

**Not covered:** PRIN 2022, PRIN 2022 PNRR and PRIN 2024 are not in the
portal search (their funded lists are PDF decree annexes on mur.gov.it) — a
follow-up source. They are the most-cited PRIN calls, so they stay as
citation-derived shells for now.

**Prerequisites:**
- Run `scripts/local/prin_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/prin/prin_projects.parquet` (§1.4 shrink guard).

**Funder routing (§2.3.2 — the source spans two ministries):** PRIN was
funded by **MIUR** (Ministero dell'Istruzione, dell'Università e della
Ricerca, F4320321873) for the calls up to 2017 and by **MUR** (Ministero
dell'Università e della Ricerca, F4320331528, re-established 2020) from PRIN
2020. Rows are routed on the call year: `< 2020 → MIUR`, `>= 2020 → MUR`.
Caveats: the 1999–2000 calls were technically MURST and 2006–2007 fell in the
2006–2008 MUR interlude; neither historical entity has its own OpenAlex funder
(F4320331528 is the 2020 ministry, DOI 10.13039/501100021856), and citing
works credit those calls to MIUR, so they route to MIUR.

**`funder_award_id` (§2.1.1):** the protocol code (`2017FJCPEX`,
`20158KBFTY`, `9910034148`). Citation-side check (openalex_awards_raw shells
for F4320321873 / F4320331528): researchers cite the bare code, often with a
`prot.`/`PRIN` prefix or a `_00N` unit suffix (`2010BX2SNA_002`); ~1.8k
distinct 2017 codes and ~390 2020 codes already exist as priority-0 shells.
The source has only one identifier.

**Dates:** the portal publishes the call year and duration (months) but no
start/end dates, so `start_date`/`end_date` are NULL; `start_year` = call year
(first year of `2010-2011`).

**Funder details:** read by ID from `openalex.funders.funders` (both F4320*).

**Priority:** `494` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.prin_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/prin/prin_projects.parquet`;

In [ ]:
%sql
SELECT call_year, COUNT(*) as n FROM openalex.awards.prin_raw GROUP BY call_year ORDER BY call_year;

In [ ]:
%sql
DESCRIBE openalex.awards.prin_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.prin_raw LIMIT 5;

## Step 1.6: Funder existence check
Both funders are Crossref-registered (F4320*), so each MUST resolve to exactly
1 row in `openalex.funders.funders`. If not, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id IN (4320331528, 4320321873);

## Step 2: Create PRIN Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.prin_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id IN (4320331528, 4320321873)
),

parsed AS (
    SELECT
        g.*,
        TRY_CAST(LEFT(g.call_year, 4) AS INT) AS call_year_int,
        -- §2.3.2 routing on the call year: MUR (2020+) vs MIUR (up to 2017)
        CASE WHEN TRY_CAST(LEFT(g.call_year, 4) AS INT) >= 2020 THEN 4320331528
             ELSE 4320321873 END AS resolved_funder_id,
        from_json(g.units, 'ARRAY<STRUCT<unit:INT, institution:STRING, name:STRING, given_name:STRING, family_name:STRING, amount:DOUBLE>>') AS unit_arr
    FROM openalex.awards.prin_raw g
    WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
),

people AS (
    -- Research units in portal order (unit 1 = coordinator) -> investigator structs.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(COALESCE(unit_arr, ARRAY()), u -> named_struct(
            'given_name', NULLIF(TRIM(u.given_name), ''),
            'family_name', NULLIF(TRIM(u.family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(u.institution), ''),
                'country', 'IT',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        )) AS investigators
    FROM parsed
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'EUR' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CONCAT('PRIN ', g.call_year) as funder_scheme,
    'mur_prin_portal' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    g.call_year_int as start_year,
    CAST(NULL AS INT) as end_year,
    -- lead = the coordinator from the header (same person as unit 1)
    named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', 'IT',
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM parsed g
JOIN people p ON p.funder_award_id = g.funder_award_id
JOIN src_funder f ON f.funder_id = g.resolved_funder_id;

In [ ]:
%sql
-- funder_award_id shape check (§2.1.1): every id is a PRIN protocol code
-- (10 alphanumerics: 4-digit year + 6, or the 1999-2001 all-digit form).
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9A-Z]{10}$' THEN 1 ELSE 0 END) as protocol_shaped,
    COLLECT_SET(CASE WHEN NOT funder_award_id RLIKE '^[0-9A-Z]{10}$' THEN funder_award_id END) as odd_ids
FROM openalex.awards.prin_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mur_prin_portal' AND priority = 494;

-- Insert into openalex_awards_raw with priority.
-- Priority 494: direct-from-funder ingest of the ministry's PRIN portal.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 494 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    494 as priority
FROM openalex.awards.prin_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_prin_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.prin_awards;

In [ ]:
%sql
-- §6.5 funder split: MIUR for calls <= 2017, MUR for 2020.
SELECT funder.display_name, funder_id, MIN(start_year) as first_call, MAX(start_year) as last_call, COUNT(*) as n
FROM openalex.awards.prin_awards
GROUP BY ALL ORDER BY n DESC;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, size(investigators) as n_units
FROM openalex.awards.prin_awards LIMIT 20;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.prin_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.prin_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.prin_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.prin_awards;

In [ ]:
%sql
-- Overlap with existing citation shells: how many shells this ingest upgrades.
SELECT c.funder_id, COUNT(DISTINCT c.id) as shell_ids, COUNT(DISTINCT t.id) as upgraded_by_prin
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.prin_awards t ON t.id = c.id
WHERE c.funder_id IN (4320331528, 4320321873) AND c.priority = 0
GROUP BY c.funder_id;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, funder_id, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mur_prin_portal'
GROUP BY ALL;